# LeetCode 1648 — Sell Diminishing-Valued Colored Balls bằng Python

Link đề: https://leetcode.com/problems/sell-diminishing-valued-colored-balls/ · Độ khó: **Medium** · Chủ đề: Array, Math, Binary Search, Greedy, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maxProfit(self, inventory: List[int], orders: int) -> int:
        ...
```

Notebook đi từ **mô phỏng từng quả bóng** (quét tìm màu nhiều nhất) → **heap** → **heap bán theo lô** → **sắp xếp + quét theo tầng** → **tìm kiếm nhị phân ngưỡng giá** `T`. Trọng tâm là chuyển từ "bán từng quả" (tới `10⁹` lần) sang "bán cả một **cấp số cộng** trong một phép tính".

## 1. Mục tiêu học

1. Hiểu nhận xét tham lam: **mỗi lần bán, hãy bán quả có giá cao nhất hiện tại** (màu đang còn nhiều nhất).
2. Biết vì sao mô phỏng từng quả (`orders` tới `10⁹`) là quá chậm, và cách gom nhiều lần bán thành **tổng cấp số cộng** `l + (l+1) + … + r`.
3. Cài đặt hai lời giải `O(n log n)`: sắp xếp + quét tầng, và tìm kiếm nhị phân **ngưỡng giá** `T`.
4. Xử lý số lớn và **modulo `10⁹ + 7`** đúng chỗ (chia 2 trước khi lấy mod, hoặc tính chính xác bằng số nguyên Python rồi mới mod).
5. Đối chiếu mọi cách với mô phỏng trên input nhỏ, seed cố định.

## 2. Đề bài & ví dụ

Bạn có một kho bóng nhiều màu. Một khách hàng muốn mua bóng màu bất kỳ, và định giá rất lạ: **giá của một quả bóng = số quả cùng màu bạn đang có trong kho** (ngay trước khi bán). Ví dụ có 6 quả vàng, quả vàng đầu tiên bán được 6; còn 5 quả, quả vàng tiếp theo bán được 5; …

Cho mảng `inventory` với `inventory[i]` = số bóng màu `i` ban đầu, và số nguyên `orders` = tổng số bóng khách muốn mua. Bạn được bán theo **thứ tự tùy ý**. Trả về **tổng giá trị lớn nhất** thu được sau khi bán đúng `orders` quả, **modulo `10⁹ + 7`**.

**Ví dụ 1:** `inventory = [2,5]`, `orders = 4` → `14`. Bán màu 1 một lần (2), màu 2 ba lần (5 + 4 + 3): `2 + 5 + 4 + 3 = 14`.

**Ví dụ 2:** `inventory = [3,5]`, `orders = 6` → `19`. Bán màu 1 hai lần (3 + 2), màu 2 bốn lần (5 + 4 + 3 + 2): tổng `19`.

**Ràng buộc:**
- `1 ≤ inventory.length ≤ 10⁵`
- `1 ≤ inventory[i] ≤ 10⁹`
- `1 ≤ orders ≤ min(sum(inventory[i]), 10⁹)`

In [1]:
import heapq
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [([2, 5], 4, 14), ([3, 5], 6, 19)]

def sell_sequence(inventory, orders):
    """Mô phỏng tham lam, trả về danh sách giá từng lần bán (chỉ dùng cho input nhỏ)."""
    inv = list(inventory)
    prices = []
    for _ in range(orders):
        k = max(range(len(inv)), key=lambda i: inv[i])
        prices.append(inv[k]); inv[k] -= 1
    return prices

for inv, orders, exp in examples:
    seq = sell_sequence(inv, orders)
    print(f"inventory={inv}, orders={orders}: bán theo giá {seq} -> tổng {sum(seq)} (kỳ vọng {exp})")

inventory=[2, 5], orders=4: bán theo giá [5, 4, 3, 2] -> tổng 14 (kỳ vọng 14)
inventory=[3, 5], orders=6: bán theo giá [5, 4, 3, 3, 2, 2] -> tổng 19 (kỳ vọng 19)


## 3. Kiến thức nền

### 3.1 Tổng cấp số cộng

Tổng các số nguyên liên tiếp từ `l` tới `r` (với `l ≤ r`):

`l + (l+1) + … + r = (l + r) · (r − l + 1) / 2`

Tích `(l + r)(r − l + 1)` luôn **chẵn** (một trong hai thừa số chẵn), nên phép chia nguyên `//` là chính xác.

In [2]:
def range_sum(l, r):
    """Tổng l + (l+1) + ... + r; bằng 0 nếu l > r."""
    if l > r:
        return 0
    return (l + r) * (r - l + 1) // 2

print(range_sum(3, 5), "=", 3 + 4 + 5)
print(range_sum(1, 100), "=", sum(range(1, 101)))
print(range_sum(10**9 - 5, 10**9), "(số lớn, Python tính chính xác)")

12 = 12
5050 = 5050
5999999985 (số lớn, Python tính chính xác)


### 3.2 Số nguyên lớn và modulo

Python có **số nguyên vô hạn**, nên ta có thể tính **chính xác** rồi `% MOD` một lần ở cuối. Nếu muốn lấy mod giữa chừng, **không** được lấy mod trước khi **chia 2** (vì phép chia không "tương thích" với mod theo cách đơn giản).

In [3]:
l, r = 10**9 - 3, 10**9
exact = range_sum(l, r)
print("chính xác % MOD        :", exact % MOD)
bad = ((l + r) % MOD) * ((r - l + 1) % MOD) % MOD // 2
print("mod rồi mới chia 2 (sai):", bad)
good = (l + r) * (r - l + 1) // 2 % MOD
print("chia 2 rồi mới mod     :", good)

chính xác % MOD        : 999999973
mod rồi mới chia 2 (sai): 499999969
chia 2 rồi mới mod     : 999999973


### 3.3 Max-heap bằng `heapq`

Đẩy `-x` để `heap[0]` cho `-max`. Cập nhật phần tử lớn nhất: `heapq.heapreplace(heap, -new)` (lấy ra rồi đẩy vào trong một lần).

In [4]:
h = [-x for x in [2, 5, 3]]
heapq.heapify(h)
print("lớn nhất:", -h[0])
heapq.heapreplace(h, -4)        # 5 -> 4
print("sau khi giảm màu lớn nhất xuống 4:", sorted(-x for x in h))

lớn nhất: 5
sau khi giảm màu lớn nhất xuống 4: [2, 3, 4]


### 3.4 Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Nếu `ok(T)` **đơn điệu** (sai … sai đúng … đúng khi `T` tăng), ta tìm `T` nhỏ nhất với `ok(T)` đúng trong `O(log(range))` lần gọi.

In [5]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda t: t * t >= 50))

8


## 4. Ý tưởng nền tảng / trực giác

Tưởng tượng mỗi màu là một **cột** cao `inventory[i]`; quả bóng ở tầng `h` (tính từ dưới, 1-indexed) có giá `h`. Bán một quả = **gỡ quả trên cùng** của một cột, nhận giá bằng chiều cao cột lúc đó.

Ví dụ 2 `inventory = [3, 5]`, `orders = 6`:

```
tầng 5:      ■
tầng 4:      ■
tầng 3:  ■   ■
tầng 2:  ■   ■
tầng 1:  ■   ■
```

Muốn tổng lớn nhất ⇒ lấy `6` quả ở **các tầng cao nhất**: tầng 5 (1 quả), tầng 4 (1), tầng 3 (2), tầng 2 (2) ⇒ `5 + 4 + 3·2 + 2·2 = 19`.

Ta luôn gỡ từ **trên xuống** — "đường cắt ngang" hạ dần. Toàn bộ bài toán là: tìm **mức cắt `T`** sao cho số quả trên mức `T` vừa khớp `orders`, phần lẻ lấy ở đúng tầng `T`.

In [6]:
def draw(inventory, cut=None):
    top = max(inventory)
    for level in range(top, 0, -1):
        row = "".join("  ■ " if h >= level else "    " for h in inventory)
        mark = "  <- mức cắt T" if cut == level else ""
        print(f"tầng {level:>2}:{row}{mark}")

draw([3, 5], cut=2)
print("Số quả ở tầng > 2:", sum(max(0, h - 2) for h in [3, 5]), "| phần lẻ ở tầng 2:", 6 - sum(max(0, h - 2) for h in [3, 5]))

tầng  5:      ■ 
tầng  4:      ■ 
tầng  3:  ■   ■ 
tầng  2:  ■   ■   <- mức cắt T
tầng  1:  ■   ■ 
Số quả ở tầng > 2: 4 | phần lẻ ở tầng 2: 2


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng quả

**Ý tưởng:** lặp `orders` lần; mỗi lần quét tìm màu còn nhiều nhất, cộng giá, giảm 1.

In [7]:
def max_profit_brute(inventory, orders):
    inv = list(inventory)
    total = 0
    for _ in range(orders):
        k = 0
        for i in range(1, len(inv)):
            if inv[i] > inv[k]:
                k = i
        total += inv[k]
        inv[k] -= 1
    return total % MOD

for inv, orders, exp in examples:
    print(max_profit_brute(inv, orders), "(kỳ vọng", exp, ")")

14 (kỳ vọng 14 )
19 (kỳ vọng 19 )


**Độ phức tạp:** `O(orders · n)` — với `orders = 10⁹`, `n = 10⁵` là `10¹⁴` phép tính. Không khả thi.

**Nút thắt 1:** quét tìm max mỗi lần (`O(n)`) — có thể thay bằng heap `O(log n)`.
**Nút thắt 2 (quan trọng hơn):** số lần bán `orders` quá lớn — phải bán **cả lô** trong một phép tính.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam:** luôn bán quả đắt nhất hiện có. (Hoán đổi: nếu một phương án bán quả giá `x` mà bỏ lại quả giá `y > x` không bán, thay bằng quả `y` chỉ tăng tổng — chi tiết ở mục 10.)

**Gom lô:** khi các màu lớn nhất cùng ở mức `top` và màu kế tiếp ở mức `nxt < top`, ta bán **liền một lèo** từ `top` xuống `nxt + 1` cho tất cả `cnt` màu đó: tổng `cnt · range_sum(nxt + 1, top)`, tốn `cnt · (top − nxt)` lượt.
- **Nếu** `cnt · (top − nxt) ≤ orders` **thì** bán hết lô, các màu này nhập vào nhóm ở mức `nxt`.
- **Ngược lại** chỉ bán được `full = orders // cnt` tầng đầy, cộng thêm `orders % cnt` quả ở tầng `top − full`.

**Ngưỡng `T`:** gọi `above(T) = Σ max(0, a_i − T)` = số quả ở tầng **cao hơn** `T`. Hàm này **giảm** khi `T` tăng. Tìm `T` nhỏ nhất sao cho `above(T) ≤ orders`: bán hết các quả trên `T`, phần còn lại `orders − above(T)` quả bán ở giá `T`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng bằng max-heap

In [8]:
def max_profit_heap(inventory, orders):
    heap = [-x for x in inventory]
    heapq.heapify(heap)
    total = 0
    for _ in range(orders):
        top = -heap[0]
        total += top
        heapq.heapreplace(heap, -(top - 1))
    return total % MOD

for inv, orders, exp in examples:
    print(max_profit_heap(inv, orders), "(kỳ vọng", exp, ")")

14 (kỳ vọng 14 )
19 (kỳ vọng 19 )


**Độ phức tạp:** `O(n + orders · log n)` — vẫn bị `orders = 10⁹` "giết". Nhưng đây là bước trung gian tốt để kiểm tra tham lam.

### 7.2 Cách 3 — Heap bán theo lô

Gom các màu **cùng mức cao nhất** vào một nhóm `(mức, số_màu)`. Mỗi bước: lấy nhóm cao nhất, nhìn mức kế tiếp `nxt` (hoặc `0`), bán theo lô như mục 6, rồi gộp nhóm vào mức `nxt`.

In [9]:
from collections import Counter

def max_profit_heap_batch(inventory, orders):
    cnt_by_level = Counter(inventory)
    heap = [-v for v in cnt_by_level]          # các mức khác nhau, max-heap
    heapq.heapify(heap)
    total = 0
    group = 0                                  # số màu đang ở mức cao nhất
    while orders > 0:
        top = -heapq.heappop(heap)
        group += cnt_by_level[top]
        nxt = -heap[0] if heap else 0
        layers = top - nxt
        if group * layers <= orders:
            total += group * range_sum(nxt + 1, top)
            orders -= group * layers
            # nhóm sẽ gộp với mức nxt ở vòng sau (cnt_by_level[nxt] cộng thêm khi pop)
        else:
            full, rest = divmod(orders, group)
            total += group * range_sum(top - full + 1, top) + rest * (top - full)
            orders = 0
    return total % MOD

for inv, orders, exp in examples:
    print(max_profit_heap_batch(inv, orders), "(kỳ vọng", exp, ")")

14 (kỳ vọng 14 )
19 (kỳ vọng 19 )


Lưu ý: `group` **cộng dồn** — sau khi hạ nhóm xuống mức `nxt`, ở vòng sau ta `pop` mức `nxt` và cộng thêm số màu vốn ở mức đó. Nếu `heap` rỗng thì `nxt = 0` (bán tới tận đáy). Vì `orders ≤ sum(inventory)` nên vòng lặp luôn kết thúc trước khi hết bóng.

**Độ phức tạp:** mỗi mức khác nhau được pop một lần ⇒ `O(n log n)`, bộ nhớ `O(n)`.

### 7.3 Cách 4 — Sắp xếp giảm dần + quét tầng

Giống cách 3 nhưng không cần heap: sắp `inventory` giảm dần, thêm phần tử `0` ở cuối làm "đáy". Duyệt `i` từ `0`; nhóm cao nhất là `inv[0..i]` (tất cả đã bị hạ về mức `inv[i]`), mức kế tiếp là `inv[i+1]`.

In [10]:
def max_profit_sorted(inventory, orders):
    inv = sorted(inventory, reverse=True) + [0]
    total = 0
    for i in range(len(inv) - 1):
        top, nxt = inv[i], inv[i + 1]
        if top == nxt:
            continue                          # chưa có chênh lệch, nhóm lớn dần
        group = i + 1
        layers = top - nxt
        if group * layers <= orders:
            total += group * range_sum(nxt + 1, top)
            orders -= group * layers
        else:
            full, rest = divmod(orders, group)
            total += group * range_sum(top - full + 1, top) + rest * (top - full)
            orders = 0
        if orders == 0:
            break
    return total % MOD

for inv, orders, exp in examples:
    print(max_profit_sorted(inv, orders), "(kỳ vọng", exp, ")")

14 (kỳ vọng 14 )
19 (kỳ vọng 19 )


In [11]:
class SolutionSorted:
    def maxProfit(self, inventory: List[int], orders: int) -> int:
        inv = sorted(inventory, reverse=True) + [0]
        total = 0
        for i in range(len(inv) - 1):
            top, nxt = inv[i], inv[i + 1]
            if top == nxt:
                continue
            group, layers = i + 1, top - nxt
            if group * layers <= orders:
                total += group * (top + nxt + 1) * layers // 2
                orders -= group * layers
            else:
                full, rest = divmod(orders, group)
                total += group * (top + top - full + 1) * full // 2 + rest * (top - full)
                orders = 0
            if orders == 0:
                break
        return total % (10**9 + 7)

print([SolutionSorted().maxProfit(inv, o) for inv, o, _ in examples])

[14, 19]


**Độ phức tạp:** sắp xếp `O(n log n)` + quét `O(n)`; bộ nhớ `O(n)`.

### 7.4 Cách 5 — Tìm kiếm nhị phân ngưỡng giá `T`

1. `above(T) = Σ max(0, a − T)`.
2. Tìm `T` nhỏ nhất trong `[0, max(inventory)]` với `above(T) ≤ orders` (`above(max) = 0` nên luôn tồn tại).
3. Tổng = `Σ_{a > T} range_sum(T + 1, a)` + `(orders − above(T)) · T`.

Phần lẻ `orders − above(T)` luôn **nhỏ hơn** số màu có `a ≥ T` (vì `above(T − 1) > orders`), nên thật sự có đủ quả ở tầng `T` để bán.

In [12]:
def max_profit_binary(inventory, orders):
    def above(T):
        return sum(a - T for a in inventory if a > T)
    lo, hi = 0, max(inventory)
    while lo < hi:
        mid = (lo + hi) // 2
        if above(mid) <= orders:
            hi = mid
        else:
            lo = mid + 1
    T = lo
    total = sum(range_sum(T + 1, a) for a in inventory if a > T)
    total += (orders - above(T)) * T
    return total % MOD

for inv, orders, exp in examples:
    print(max_profit_binary(inv, orders), "(kỳ vọng", exp, ")")

14 (kỳ vọng 14 )
19 (kỳ vọng 19 )


**Độ phức tạp:** `O(n · log(max))` ≈ `10⁵ · 30` — rất nhanh, không cần sắp xếp; bộ nhớ `O(1)`. Đây là lời giải nộp chính (mục 9).

## 8. TRACE từng bước

### 8.1 Trace cách 4 (quét tầng) trên `inventory = [2, 8, 4, 10, 6]`, `orders = 20` (ví dụ tự dựng)

Ví dụ tự dựng; đáp án được đối chiếu bằng mô phỏng (nên đối chiếu lại đề gốc nếu muốn). Sau khi sắp: `[10, 8, 6, 4, 2, 0]`.

| i | top | nxt | nhóm | lượt cần | orders trước | hành động | cộng thêm |
|---|---|---|---|---|---|---|---|
| 0 | 10 | 8 | 1 | 2 | 20 | bán hết | 10+9 = 19 |
| 1 | 8 | 6 | 2 | 4 | 18 | bán hết | 2·(8+7) = 30 |
| 2 | 6 | 4 | 3 | 6 | 14 | bán hết | 3·(6+5) = 33 |
| 3 | 4 | 2 | 4 | 8 | 8 | bán hết | 4·(4+3) = 28 |
| — | | | | | 0 | dừng | tổng 110 |

In [13]:
def trace_sorted(inventory, orders):
    inv = sorted(inventory, reverse=True) + [0]
    total = 0
    print(f"sắp giảm dần + đáy 0: {inv}, orders={orders}")
    for i in range(len(inv) - 1):
        top, nxt = inv[i], inv[i + 1]
        if top == nxt:
            print(f"  i={i}: top={top} == nxt -> nhóm lớn thêm, chưa bán"); continue
        group, layers = i + 1, top - nxt
        need = group * layers
        if need <= orders:
            add = group * range_sum(nxt + 1, top)
            orders -= need
            print(f"  i={i}: top={top}, nxt={nxt}, nhóm={group}, cần {need} lượt -> bán hết tầng {nxt + 1}..{top}: +{add}, còn orders={orders}")
        else:
            full, rest = divmod(orders, group)
            add = group * range_sum(top - full + 1, top) + rest * (top - full)
            print(f"  i={i}: top={top}, nxt={nxt}, nhóm={group}, cần {need} > {orders} -> {full} tầng đầy + {rest} quả giá {top - full}: +{add}")
            orders = 0
        total += add
        if orders == 0:
            break
    print("  tổng =", total)
    return total

t = trace_sorted([2, 8, 4, 10, 6], 20)
print("mô phỏng:", sum(sell_sequence([2, 8, 4, 10, 6], 20)))

sắp giảm dần + đáy 0: [10, 8, 6, 4, 2, 0], orders=20
  i=0: top=10, nxt=8, nhóm=1, cần 2 lượt -> bán hết tầng 9..10: +19, còn orders=18
  i=1: top=8, nxt=6, nhóm=2, cần 4 lượt -> bán hết tầng 7..8: +30, còn orders=14
  i=2: top=6, nxt=4, nhóm=3, cần 6 lượt -> bán hết tầng 5..6: +33, còn orders=8
  i=3: top=4, nxt=2, nhóm=4, cần 8 lượt -> bán hết tầng 3..4: +28, còn orders=0
  tổng = 110
mô phỏng: 110


In [14]:
t = trace_sorted([3, 5], 6)
print()
t = trace_sorted([7, 7, 7], 8)
print("mô phỏng [7,7,7], 8:", sell_sequence([7, 7, 7], 8))

sắp giảm dần + đáy 0: [5, 3, 0], orders=6
  i=0: top=5, nxt=3, nhóm=1, cần 2 lượt -> bán hết tầng 4..5: +9, còn orders=4
  i=1: top=3, nxt=0, nhóm=2, cần 6 > 4 -> 2 tầng đầy + 0 quả giá 1: +10
  tổng = 19

sắp giảm dần + đáy 0: [7, 7, 7, 0], orders=8
  i=0: top=7 == nxt -> nhóm lớn thêm, chưa bán
  i=1: top=7 == nxt -> nhóm lớn thêm, chưa bán
  i=2: top=7, nxt=0, nhóm=3, cần 21 > 8 -> 2 tầng đầy + 2 quả giá 5: +49
  tổng = 49
mô phỏng [7,7,7], 8: [7, 7, 7, 6, 6, 6, 5, 5]


### 8.2 Trace cách 5 (tìm kiếm nhị phân) trên ví dụ 1 và 2

In [15]:
def trace_binary(inventory, orders):
    def above(T):
        return sum(a - T for a in inventory if a > T)
    lo, hi = 0, max(inventory)
    print(f"inventory={inventory}, orders={orders}")
    while lo < hi:
        mid = (lo + hi) // 2
        a = above(mid)
        print(f"   lo={lo}, hi={hi}, mid={mid}: above(mid)={a} {'<=' if a <= orders else '>'} {orders} -> {'hi=mid' if a <= orders else 'lo=mid+1'}")
        if a <= orders:
            hi = mid
        else:
            lo = mid + 1
    T = lo
    part1 = sum(range_sum(T + 1, a) for a in inventory if a > T)
    rest = orders - above(T)
    print(f"   T={T}: bán hết {above(T)} quả trên T (tổng {part1}) + {rest} quả giá {T} -> {part1 + rest * T}")
    return part1 + rest * T

trace_binary([2, 5], 4)
trace_binary([3, 5], 6)

inventory=[2, 5], orders=4
   lo=0, hi=5, mid=2: above(mid)=3 <= 4 -> hi=mid
   lo=0, hi=2, mid=1: above(mid)=5 > 4 -> lo=mid+1
   T=2: bán hết 3 quả trên T (tổng 12) + 1 quả giá 2 -> 14
inventory=[3, 5], orders=6
   lo=0, hi=5, mid=2: above(mid)=4 <= 6 -> hi=mid
   lo=0, hi=2, mid=1: above(mid)=6 <= 6 -> hi=mid
   lo=0, hi=1, mid=0: above(mid)=8 > 6 -> lo=mid+1
   T=1: bán hết 6 quả trên T (tổng 19) + 0 quả giá 1 -> 19


19

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maxProfit(self, inventory: List[int], orders: int) -> int:
        MOD = 10**9 + 7

        def above(T: int) -> int:                 # số quả có giá > T
            return sum(a - T for a in inventory if a > T)

        lo, hi = 0, max(inventory)
        while lo < hi:
            mid = (lo + hi) // 2
            if above(mid) <= orders:
                hi = mid
            else:
                lo = mid + 1
        T = lo
        total = 0
        for a in inventory:
            if a > T:
                total += (a + T + 1) * (a - T) // 2   # T+1 + ... + a
        total += (orders - above(T)) * T
        return total % MOD

for inv, orders, exp in examples:
    assert Solution().maxProfit(inv, orders) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tập quả bán được là cố định theo chiều cao:** mỗi màu có các quả giá `a_i, a_i − 1, …, 1` và **muốn bán quả giá `h` của màu `i` thì phải bán trước mọi quả giá `> h` của màu đó**. Bất kỳ tập `S` gồm `orders` quả thỏa ràng buộc "đóng xuống dưới" (nếu có quả giá `h` thì có mọi quả giá `> h` cùng màu) đều bán được, giá trị là tổng các nhãn trong `S`.

**Tối ưu = lấy `orders` nhãn lớn nhất:** tập gồm `orders` quả có nhãn lớn nhất (phá hòa tùy ý) **luôn** thỏa ràng buộc đóng: nếu nó chứa quả giá `h` của màu `i` mà thiếu quả giá `h' > h` cùng màu, đổi hai quả sẽ được tập có tổng lớn hơn — mâu thuẫn. Và không tập nào cỡ `orders` có tổng lớn hơn tập các nhãn lớn nhất. ⇒ đáp án = tổng `orders` nhãn lớn nhất trong đa tập `{1..a_i}`.

**Cách 5 tính đúng tổng đó:** `T` là giá trị nhỏ nhất với `above(T) ≤ orders`. Mọi `above(T)` quả có nhãn `> T` đều nằm trong top-`orders`; vì `above(T − 1) > orders`, còn thiếu `orders − above(T)` quả và chúng phải có nhãn đúng bằng `T` (có `above(T − 1) − above(T)` quả nhãn `T`, đủ dùng). Tổng = `Σ_{a>T} (T+1 + … + a) + (orders − above(T)) · T`.

**Cách 3, 4 (quét tầng):** chính là hạ "đường cắt" từ trên xuống qua từng mức có trong `inventory`, mỗi lần lấy trọn các tầng giữa hai mức liên tiếp — cũng là lấy các nhãn lớn nhất theo thứ tự giảm dần.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(inventory)`, `M = max(inventory)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng quét | `O(orders · n)` | `O(n)` | chỉ cho input nhỏ |
| 2. Max-heap từng quả | `O(n + orders · log n)` | `O(n)` | `orders` lớn ⇒ quá chậm |
| 3. Heap theo lô | `O(n log n)` | `O(n)` | mỗi mức pop một lần |
| 4. Sắp xếp + quét tầng | `O(n log n)` | `O(n)` | |
| 5. Nhị phân ngưỡng `T` | `O(n log M)` | `O(1)` | `log M ≈ 30` |

Mọi phép nhân trung gian có thể tới `~10⁵ · 10¹⁸` — Python xử lý được nhờ số nguyên lớn. Trong C++/Java phải lấy mod cẩn thận (và chia 2 trước khi mod).

In [17]:
rnd = random.Random(1648)
inv = [rnd.randint(1, 10**9) for _ in range(10**5)]
orders = min(sum(inv), 10**9)
for name, fn in [("heap_batch", max_profit_heap_batch), ("sorted", max_profit_sorted), ("binary", max_profit_binary), ("Solution", Solution().maxProfit)]:
    t0 = time.perf_counter(); r = fn(list(inv), orders); dt = time.perf_counter() - t0
    print(f"{name:<11}: {r:>10}  {dt * 1000:7.1f} ms")

heap_batch :  500993370     16.2 ms
sorted     :  500993370     15.0 ms
binary     :  500993370     50.7 ms


Solution   :  500993370     50.0 ms


## 12. Bộ test

Đối chiếu với mô phỏng (cách 1, 2) trên input nhỏ; edge case (một màu, mọi màu bằng nhau, bán hết toàn bộ, `orders = 1`); kiểm tra **giá trị chính xác** (không mod) giữa các cách và với "top-`orders` nhãn lớn nhất" sinh trực tiếp.

In [18]:
ALL = {
    "brute": max_profit_brute,
    "heap": max_profit_heap,
    "heap_batch": max_profit_heap_batch,
    "sorted": max_profit_sorted,
    "binary": max_profit_binary,
    "SolSorted": lambda inv, o: SolutionSorted().maxProfit(list(inv), o),
    "Solution": lambda inv, o: Solution().maxProfit(list(inv), o),
}

def top_labels(inventory, orders):
    """Định nghĩa trực tiếp: tổng orders nhãn lớn nhất trong đa tập {1..a_i}."""
    labels = sorted((h for a in inventory for h in range(1, a + 1)), reverse=True)
    return sum(labels[:orders]) % MOD

def check(inventory, orders, exp=None, small=True):
    ref = top_labels(inventory, orders) if small else max_profit_sorted(inventory, orders)
    for name, fn in ALL.items():
        if not small and name in ("brute", "heap"):
            continue
        got = fn(list(inventory), orders)
        assert got == ref, (name, inventory, orders, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for inv, orders, exp in examples:
    check(inv, orders, exp)
print("Ví dụ đề: OK")

check([1], 1, 1)
check([10], 10, 55)                 # bán hết một màu
check([5, 5, 5], 15, 45)            # bán hết toàn bộ
check([5, 5, 5], 4, 19)             # 5+5+5+4
check([1000000000], 1, 10**9 % MOD, small=False)        # nhãn quá nhiều để liệt kê: so giữa các cách nhanh
check([2, 8, 4, 10, 6], 20, 110)
check([497978859, 167261111, 483575207, 591815159], 836556809, small=False)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [19]:
rnd = random.Random(2021)
for _ in range(2500):
    n = rnd.randint(1, 6)
    inv = [rnd.randint(1, 12) for _ in range(n)]
    orders = rnd.randint(1, sum(inv))
    check(inv, orders)
print("2500 bộ nhỏ so với định nghĩa trực tiếp: OK")

for _ in range(200):
    n = rnd.randint(1, 300)
    inv = [rnd.randint(1, 10**9) for _ in range(n)]
    orders = rnd.randint(1, min(sum(inv), 10**9))
    check(inv, orders, small=False)
print("200 bộ cỡ lớn so giữa các cách nhanh: OK")

2500 bộ nhỏ so với định nghĩa trực tiếp: OK
200 bộ cỡ lớn so giữa các cách nhanh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bán cạn từng màu một (màu lớn nhất trước)

In [20]:
def max_profit_bug_exhaust(inventory, orders):
    total = 0
    for a in sorted(inventory, reverse=True):
        take = min(a, orders)
        total += range_sum(a - take + 1, a)          # SAI: bán cạn màu này trước
        orders -= take
        if orders == 0:
            break
    return total % MOD

for inv, o in [([3, 3], 2), ([5, 4], 4)]:
    print(f"{inv}, orders={o}: sai = {max_profit_bug_exhaust(inv, o)}, đúng = {Solution().maxProfit(inv, o)}")

[3, 3], orders=2: sai = 5, đúng = 6
[5, 4], orders=4: sai = 14, đúng = 16


**Vì sao sai:** sau khi bán quả đầu tiên của màu `3`, quả tiếp theo của màu đó chỉ còn giá `2`, trong khi màu kia vẫn có quả giá `3`. Tham lam đúng là **mỗi lần** chọn quả đắt nhất — tức hạ "đường cắt" **đồng đều** trên mọi cột. **Sửa:** bán theo tầng (cách 4) hoặc theo ngưỡng `T` (cách 5).

### Lỗi 2 — Lấy mod trước khi chia 2

In [21]:
def range_sum_bug_mod(l, r):
    return ((l + r) % MOD) * ((r - l + 1) % MOD) % MOD // 2      # SAI: chia 2 sau khi mod

def max_profit_bug_mod(inventory, orders):
    def above(T):
        return sum(a - T for a in inventory if a > T)
    lo, hi = 0, max(inventory)
    while lo < hi:
        mid = (lo + hi) // 2
        if above(mid) <= orders: hi = mid
        else: lo = mid + 1
    T = lo
    total = 0
    for a in inventory:
        if a > T:
            total = (total + range_sum_bug_mod(T + 1, a)) % MOD
    return (total + (orders - above(T)) * T) % MOD

inv, o = [1000000000], 1000000000 - 9          # T = 9: cần tổng 10 + 11 + ... + 10^9
print("sai :", max_profit_bug_mod(inv, o), "| đúng:", Solution().maxProfit(inv, o))

sai : 499999979 | đúng: 999999983


**Vì sao sai:** `(x mod p) // 2` khác `(x // 2) mod p` khi `x ≥ p`. **Sửa:** tính chính xác (Python) rồi mới mod; trong ngôn ngữ khác thì chia 2 cho thừa số chẵn **trước** khi nhân và mod, hoặc nhân với nghịch đảo modulo của 2.

### Lỗi 3 — Quên phần lẻ (hoặc bán phần lẻ ở sai giá)

In [22]:
def max_profit_bug_rest(inventory, orders):
    def above(T):
        return sum(a - T for a in inventory if a > T)
    lo, hi = 0, max(inventory)
    while lo < hi:
        mid = (lo + hi) // 2
        if above(mid) <= orders: hi = mid
        else: lo = mid + 1
    T = lo
    total = sum(range_sum(T + 1, a) for a in inventory if a > T)
    total += (orders - above(T)) * (T + 1)          # SAI: phần lẻ phải ở giá T
    return total % MOD

inv, o = [2, 5], 4                              # T = 2, phần lẻ 1 quả
print("sai :", max_profit_bug_rest(inv, o), "| đúng:", Solution().maxProfit(inv, o))

sai : 15 | đúng: 14


**Vì sao sai:** mọi quả giá `T + 1` đã được bán hết trong phần "trên `T`"; quả còn lại tiếp theo có giá đúng `T`. **Sửa:** `(orders − above(T)) * T`.

### Lỗi 4 — Đếm sai số hạng của cấp số cộng

In [23]:
def range_sum_bug_count(l, r):
    return (l + r) * (r - l) // 2                    # SAI: số số hạng là r - l + 1

print("tổng 3..5 sai:", range_sum_bug_count(3, 5), "| đúng:", range_sum(3, 5))
print("tổng 5..5 sai:", range_sum_bug_count(5, 5), "| đúng:", range_sum(5, 5))

tổng 3..5 sai: 8 | đúng: 12
tổng 5..5 sai: 0 | đúng: 5


**Sửa:** số số hạng từ `l` tới `r` là `r − l + 1`. Một mẹo nhớ: thử ngay với `l = r` (phải ra chính `l`).

### Lỗi 5 — Mô phỏng từng quả với `orders` lớn

Code đúng về logic nhưng chạy `10⁹` vòng ⇒ **Time Limit Exceeded**. Ta không chạy thật ở đây (sẽ treo notebook); chỉ ước lượng:

In [24]:
t0 = time.perf_counter()
max_profit_heap([10**9] * 3, 200000)
dt = time.perf_counter() - t0
print(f"heap từng quả: 2·10^5 lượt mất {dt * 1000:.0f} ms -> 10^9 lượt ước tính ~{dt * 5000:.0f} giây")

heap từng quả: 2·10^5 lượt mất 16 ms -> 10^9 lượt ước tính ~79 giây


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giá giảm 1 sau mỗi lần dùng" ⇒ hình dung **cột/tầng**; lấy các **tầng cao nhất**.
- `orders`/`k` tới `10⁹` ⇒ không mô phỏng từng bước; **gom lô** bằng công thức cấp số cộng.
- Hàm đếm "số phần tử > T" **đơn điệu** theo `T` ⇒ **tìm kiếm nhị phân trên giá trị** (giống LeetCode 875 Koko, 1011, 2141).
- Python: tính chính xác bằng số nguyên lớn, mod một lần ở cuối — tránh hẳn lỗi chia sau mod.
- Luôn tự kiểm công thức cấp số cộng với `l = r` và `l > r`.

## 15. Biến thể của bài

1. **Giá giảm theo `d` khác 1** (mỗi lần bán giảm `d`): tầng thành cấp số cộng công sai `d`.
2. **Chỉ được bán tối đa `c` quả mỗi màu:** chặn dưới mỗi cột ở `a_i − c`, ngưỡng `T` vẫn dùng được với `above(T) = Σ min(c, max(0, a_i − T))`.
3. **Mua thay vì bán, giá tăng dần** (muốn tổng **nhỏ nhất**): lấy các tầng **thấp nhất** — đối xứng.
4. **LeetCode 2233 — Maximum Product After K Increments:** luôn tăng phần tử nhỏ nhất, cũng gom lô hoặc heap.
5. **LeetCode 1802 — Maximum Value at a Given Index in a Bounded Array:** tìm kiếm nhị phân + tổng cấp số cộng.
6. **Top-`k` tổng của các dãy giảm** nói chung: nhị phân trên ngưỡng giá trị + đếm.

In [25]:
def max_profit_capped(inventory, orders, c):
    """Biến thể 2: mỗi màu bán tối đa c quả (giả sử orders <= tổng min(a, c))."""
    def above(T):
        return sum(min(c, max(0, a - T)) for a in inventory)
    lo, hi = 0, max(inventory)
    while lo < hi:
        mid = (lo + hi) // 2
        if above(mid) <= orders: hi = mid
        else: lo = mid + 1
    T = lo
    total = 0
    for a in inventory:
        low = max(T + 1, a - c + 1)
        total += range_sum(low, a)
    return total + (orders - above(T)) * T

def capped_brute(inventory, orders, c):
    labels = sorted((h for a in inventory for h in range(max(1, a - c + 1), a + 1)), reverse=True)
    return sum(labels[:orders])

rnd = random.Random(15)
for _ in range(1500):
    inv = [rnd.randint(1, 12) for _ in range(rnd.randint(1, 5))]
    c = rnd.randint(1, 6)
    cap_total = sum(min(a, c) for a in inv)
    o = rnd.randint(1, cap_total)
    assert max_profit_capped(inv, o, c) == capped_brute(inv, o, c)
print("Biến thể giới hạn c quả/màu: OK | ví dụ [3,5], 4, c=2 ->", max_profit_capped([3, 5], 4, 2))

Biến thể giới hạn c quả/màu: OK | ví dụ [3,5], 4, c=2 -> 14


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng quét | mỗi lượt tìm max | `O(orders·n)` | `O(n)` | kiểm thử |
| 2. Max-heap | mỗi lượt pop max | `O(orders log n)` | `O(n)` | kiểm thử, `orders` nhỏ |
| 3. Heap theo lô | gom nhóm cùng mức | `O(n log n)` | `O(n)` | khi đã có sẵn heap |
| 4. Sắp xếp + tầng | quét mức giảm dần | `O(n log n)` | `O(n)` | trực quan, phổ biến |
| 5. Nhị phân `T` | tìm mức cắt | `O(n log M)` | `O(1)` | **nộp bài**, gọn, ít lỗi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().maxProfit([4, 4], 5)` bằng bao nhiêu?

**Bài 2 (tính tay):** Với `inventory = [6, 2, 6]`, `orders = 7`, hãy tìm mức cắt `T`, số quả trên `T`, phần lẻ và tổng.

**Bài 3 (điền TODO):** Hoàn thiện `sales_per_color(inventory, orders)` trả về list số quả **bán ra của từng màu** trong một phương án tối ưu (cách 5), và kiểm tra tổng giá trị khớp.

**Bài 4 (tự viết lại):** Viết lại cách 4 (sắp xếp + tầng) **không dùng** hàm `range_sum` và tự kiểm với `Solution` trên 1000 bộ ngẫu nhiên.

**Bài 5 (mini challenge):** Biến thể "mua": giá của quả thứ `k` (tính từ 1) mua thêm của màu `i` là `b_i + k − 1` (giá **tăng**). Viết `min_cost_buy(b, orders)` trả tổng chi phí **nhỏ nhất** để mua `orders` quả, bằng tìm kiếm nhị phân; kiểm bằng heap mô phỏng.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def sales_per_color(inventory, orders):
    # TODO: tìm T như Solution; màu a > T bán a - T quả; phân phát phần lẻ cho các màu có a >= T
    pass

# Bài 4 — TODO
def max_profit_layers_manual(inventory, orders):
    # TODO: như max_profit_sorted nhưng tự viết công thức (top + nxt + 1) * layers // 2
    pass

# Bài 5 — TODO
def min_cost_buy(b, orders):
    # TODO: below(T) = sum(max(0, T - bi + 1)) = số quả có giá <= T; tìm T nhỏ nhất với below(T) >= orders
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Hai cột cao 4.
- Mức 2: Tầng 4 (2 quả), tầng 3 (2 quả), còn 1 quả ở tầng 2.
- Mức 3: `4·2 + 3·2 + 2`.

**Bài 2**
- Mức 1: `above(3) = 3 + 0 + 3 = 6 ≤ 7`, `above(2) = 8 > 7`.
- Mức 2: `T = 3`, bán hết tầng 4..6 của hai cột cao 6.
- Mức 3: `2·(4+5+6) + 1·3`.

**Bài 3**
- Mức 1: `sold[i] = max(0, a_i − T)`.
- Mức 2: Phần lẻ `r = orders − above(T)` chia cho `r` màu bất kỳ có `a_i ≥ T` (mỗi màu thêm 1).
- Mức 3: Kiểm tra `Σ sold = orders` và `Σ range_sum(a − sold + 1, a) % MOD == Solution()`.

**Bài 4**
- Mức 1: Tổng các tầng `nxt+1..top`: `(top + nxt + 1) * (top − nxt) // 2`.
- Mức 2: Phần không trọn: `full` tầng đầy từ `top` xuống, `rest` quả ở giá `top − full`.
- Mức 3: Nhớ thêm phần tử `0` vào cuối mảng đã sắp.

**Bài 5**
- Mức 1: Đối xứng: lấy các quả **rẻ nhất**.
- Mức 2: `below(T)` tăng theo `T` ⇒ tìm `T` nhỏ nhất với `below(T) ≥ orders`.
- Mức 3: Tổng = `Σ range_sum(b_i, T − 1)` cho quả giá `< T`, cộng phần còn lại ở giá `T`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
print("Bài 1:", Solution().maxProfit([4, 4], 5))
assert Solution().maxProfit([4, 4], 5) == 16 == top_labels([4, 4], 5)

# Bài 2
r2 = trace_binary([6, 2, 6], 7)
assert r2 == 2 * (4 + 5 + 6) + 3 == top_labels([6, 2, 6], 7)

# Bài 3
def sales_per_color_sol(inventory, orders):
    def above(T):
        return sum(a - T for a in inventory if a > T)
    lo, hi = 0, max(inventory)
    while lo < hi:
        mid = (lo + hi) // 2
        if above(mid) <= orders: hi = mid
        else: lo = mid + 1
    T = lo
    sold = [max(0, a - T) for a in inventory]
    rest = orders - above(T)
    for i, a in enumerate(inventory):
        if rest == 0:
            break
        if a >= T and T > 0:
            sold[i] += 1; rest -= 1
    return sold

rnd = random.Random(19)
for _ in range(1500):
    inv = [rnd.randint(1, 15) for _ in range(rnd.randint(1, 6))]
    o = rnd.randint(1, sum(inv))
    sold = sales_per_color_sol(inv, o)
    assert sum(sold) == o and all(0 <= s <= a for s, a in zip(sold, inv))
    assert sum(range_sum(a - s + 1, a) for a, s in zip(inv, sold)) % MOD == Solution().maxProfit(inv, o)
print("Bài 3 OK | ví dụ [2,5], 4 ->", sales_per_color_sol([2, 5], 4), "| [3,5], 6 ->", sales_per_color_sol([3, 5], 6))

Bài 1: 16
inventory=[6, 2, 6], orders=7
   lo=0, hi=6, mid=3: above(mid)=6 <= 7 -> hi=mid
   lo=0, hi=3, mid=1: above(mid)=11 > 7 -> lo=mid+1
   lo=2, hi=3, mid=2: above(mid)=8 > 7 -> lo=mid+1
   T=3: bán hết 6 quả trên T (tổng 30) + 1 quả giá 3 -> 33
Bài 3 OK | ví dụ [2,5], 4 -> [1, 3] | [3,5], 6 -> [2, 4]


In [28]:
# Bài 4
def max_profit_layers_manual_sol(inventory, orders):
    inv = sorted(inventory, reverse=True) + [0]
    total = 0
    for i in range(len(inv) - 1):
        top, nxt = inv[i], inv[i + 1]
        if top == nxt:
            continue
        g, layers = i + 1, top - nxt
        if g * layers <= orders:
            total += g * (top + nxt + 1) * layers // 2
            orders -= g * layers
        else:
            full, rest = divmod(orders, g)
            total += g * (2 * top - full + 1) * full // 2 + rest * (top - full)
            orders = 0
        if orders == 0:
            break
    return total % MOD

rnd = random.Random(20)
for _ in range(1000):
    inv = [rnd.randint(1, 10**6) for _ in range(rnd.randint(1, 30))]
    o = rnd.randint(1, sum(inv))
    assert max_profit_layers_manual_sol(inv, o) == Solution().maxProfit(inv, o)
print("Bài 4 OK")

# Bài 5
def min_cost_buy_sol(b, orders):
    def below(T):                       # số quả có giá <= T
        return sum(max(0, T - bi + 1) for bi in b)
    lo, hi = min(b), min(b) + orders
    while lo < hi:
        mid = (lo + hi) // 2
        if below(mid) >= orders: hi = mid
        else: lo = mid + 1
    T = lo
    cheap = below(T - 1)                # số quả giá < T
    total = sum(range_sum(bi, T - 1) for bi in b if bi <= T - 1)
    return total + (orders - cheap) * T

def min_cost_buy_heap(b, orders):
    h = list(b); heapq.heapify(h); total = 0
    for _ in range(orders):
        x = heapq.heappop(h); total += x; heapq.heappush(h, x + 1)
    return total

rnd = random.Random(21)
for _ in range(1500):
    b = [rnd.randint(1, 20) for _ in range(rnd.randint(1, 6))]
    o = rnd.randint(1, 40)
    assert min_cost_buy_sol(b, o) == min_cost_buy_heap(b, o)
print("Bài 5 OK | ví dụ b=[2,5], orders=4 ->", min_cost_buy_sol([2, 5], 4))

Bài 4 OK


Bài 5 OK | ví dụ b=[2,5], orders=4 -> 14


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lấy k nhãn lớn nhất từ các dãy a, a−1, …, 1":**

```python
def above(T): return sum(a - T for a in arr if a > T)      # số nhãn > T
lo, hi = 0, max(arr)
while lo < hi:                                             # T nhỏ nhất với above(T) <= k
    mid = (lo + hi) // 2
    if above(mid) <= k: hi = mid
    else: lo = mid + 1
T = lo
total = sum((a + T + 1) * (a - T) // 2 for a in arr if a > T)
total += (k - above(T)) * T
return total % MOD
```

**Checklist:**
- [ ] Bán theo **tầng** (đồng đều các cột), không bán cạn từng màu?
- [ ] Công thức tổng `l..r` = `(l + r)(r − l + 1) // 2` — thử với `l = r`?
- [ ] Phần lẻ bán ở giá `T` (không phải `T + 1`)?
- [ ] Chia 2 **trước** khi lấy mod (hoặc tính chính xác rồi mod)?
- [ ] Không mô phỏng từng quả khi `orders` tới `10⁹`?
- [ ] Đã đối chiếu với định nghĩa "top-`orders` nhãn" trên input nhỏ?
- [ ] Chữ ký đúng: `def maxProfit(self, inventory: List[int], orders: int) -> int`?